In [1]:
from pathlib import Path
import pandas as pd

# ===============================
# CONFIGURACIÓN
# ===============================
CARPETA = Path(r"C:\Users\luisf\IQ Tech\DashboardRotacion")
ARCHIVO = CARPETA / "rotacion_inventario_base_dashboard_odoo_autoazur.xlsx"
HOJA = "ventas_conjunto_detalle"

# ===============================
# CARGA DE DATOS
# ===============================
df = pd.read_excel(ARCHIVO, sheet_name=HOJA)
df.columns = [str(c).strip() for c in df.columns]

# ===============================
# VALIDACIONES CLAVE
# ===============================
requeridas = ["fecha", "equipo_ventas", "venta_total"]
for col in requeridas:
    if col not in df.columns:
        raise KeyError(f"No se encontró la columna obligatoria: {col}")

# ===============================
# NORMALIZACIÓN
# ===============================
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["venta_total"] = pd.to_numeric(df["venta_total"], errors="coerce").fillna(0)

if "cantidad" in df.columns:
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)

# ===============================
# FILTRO ÚLTIMOS 3 MESES
# ===============================
fecha_fin = df["fecha"].max().normalize()
fecha_inicio = fecha_fin - pd.Timedelta(days=89)

df_3m = df[
    df["fecha"].notna() &
    (df["fecha"] >= fecha_inicio) &
    (df["fecha"] <= fecha_fin)
].copy()

# ===============================
# RESUMEN POR CANAL (equipo_ventas)
# ===============================
resumen = (
    df_3m
    .groupby("equipo_ventas", as_index=False)
    .agg(
        ventas_totales=("venta_total", "sum"),
        unidades=("cantidad", "sum") if "cantidad" in df_3m.columns else ("venta_total", "size"),
        pedidos=("pedido", "nunique") if "pedido" in df_3m.columns else ("venta_total", "size"),
    )
    .sort_values("ventas_totales", ascending=False)
    .reset_index(drop=True)
)

# ===============================
# RESULTADO
# ===============================
print(resumen.to_string(index=False))

# ===============================
# EXPORTAR
# ===============================
salida = CARPETA / "ventas_totales_por_equipo_ventas_ultimos_3_meses.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="ventas_por_equipo_3m", index=False)

print(f"\nArchivo generado: {salida}")

equipo_ventas  ventas_totales  unidades  pedidos
Mercado Libre     19673510.67     16680    15710
       Amazon     14028337.69     13225    12303
      Walmart      9701804.75      4196     3734
    Liverpool      6244581.68      6932     6602
       Coppel      1833260.43       696      671
       TikTok       386109.84      1013      957

Archivo generado: C:\Users\luisf\IQ Tech\DashboardRotacion\ventas_totales_por_equipo_ventas_ultimos_3_meses.xlsx
